In [ ]:
#@title ⚙️ Setup — Run this cell first
import warnings
warnings.filterwarnings('ignore')

# ── Embedded company_sim.py ──
"""
company_sim.py — Shared infrastructure for the Regression Autopsy course.
Embedded in every notebook's setup cell. Self-contained, no external deps
beyond numpy, pandas, matplotlib, statsmodels, ipywidgets.
"""

import json
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence

import ipywidgets as widgets

# ---------------------------------------------------------------------------
# Color constants — consistent across all notebooks
# ---------------------------------------------------------------------------
COLORS = {
    'ols': '#2E5EA8',       # Blue — OLS estimator
    'truth': '#D4A017',     # Gold — true parameter
    'bias': '#C0392B',      # Red — bias/violation/error
    'repair': '#27AE60',    # Green — repair working
    'alt': '#7F8C8D',       # Gray — alternative estimators
}

# ---------------------------------------------------------------------------
# Gate System — module-level trap response storage
# ---------------------------------------------------------------------------
_trap_responses = {}
_TRAP_LOG_PATH = '/content/trap_log.json'


def _load_trap_log():
    """Load existing trap log from disk into memory."""
    global _trap_responses
    if os.path.exists(_TRAP_LOG_PATH):
        try:
            with open(_TRAP_LOG_PATH, 'r') as f:
                _trap_responses = json.load(f)
        except (json.JSONDecodeError, IOError):
            _trap_responses = {}


def _save_trap_log():
    """Persist in-memory trap responses to disk."""
    try:
        os.makedirs(os.path.dirname(_TRAP_LOG_PATH), exist_ok=True)
        with open(_TRAP_LOG_PATH, 'w') as f:
            json.dump(_trap_responses, f, indent=2)
    except IOError:
        pass  # Silently fail if /content not available (local dev)


def record_trap_response(notebook_id, question_id, response):
    """Save a trap response to the log."""
    key = f"{notebook_id}_{question_id}"
    _trap_responses[key] = {
        "response": response,
        "timestamp": datetime.now().isoformat(),
    }
    _save_trap_log()


def get_trap_response(notebook_id, question_id):
    """Retrieve a previously recorded trap response, or None."""
    key = f"{notebook_id}_{question_id}"
    entry = _trap_responses.get(key)
    return entry["response"] if entry else None


def check_gate(notebook_id, question_id):
    """Return True if a response has been recorded for this gate."""
    key = f"{notebook_id}_{question_id}"
    return key in _trap_responses


def get_all_responses():
    """Return all recorded trap responses."""
    return dict(_trap_responses)


def create_trap_widget(notebook_id, question_id, question_text, options):
    """Create a radio-button trap widget with submit button and gate logic."""
    label = widgets.HTML(f"<b>{question_text}</b>")
    radio = widgets.RadioButtons(
        options=options,
        value=None,
        layout=widgets.Layout(width='auto'),
    )
    submit = widgets.Button(description="Submit", button_style='primary')
    output = widgets.Output()

    def on_submit(btn):
        with output:
            output.clear_output()
            if radio.value is None:
                print("Please select an option before submitting.")
                return
            record_trap_response(notebook_id, question_id, radio.value)
            print(f"Response recorded: {radio.value}")
            submit.disabled = True
            radio.disabled = True

    submit.on_click(on_submit)

    # Pre-fill if already answered
    existing = get_trap_response(notebook_id, question_id)
    if existing is not None:
        try:
            radio.value = existing
            radio.disabled = True
            submit.disabled = True
        except Exception:
            pass

    return widgets.VBox([label, radio, submit, output])


# Load any existing responses at import time
_load_trap_log()


# ---------------------------------------------------------------------------
# CompanySimulator — NovaMart data generating process
# ---------------------------------------------------------------------------
class CompanySimulator:
    """
    Generates simulated NovaMart data with controllable pathologies:
    omitted variable bias, heteroscedasticity, nonlinearity, bad controls.
    """

    def __init__(
        self,
        endogeneity_strength=20,
        heteroscedasticity_strength=0.5,
        nonlinearity=True,
        bad_control_strength=0.1,
        noise_sigma=1.0,
    ):
        self.endogeneity_strength = endogeneity_strength
        self.heteroscedasticity_strength = heteroscedasticity_strength
        self.nonlinearity = nonlinearity
        self.bad_control_strength = bad_control_strength
        self.noise_sigma = noise_sigma

        # True DGP parameters
        self.beta_0 = 50
        self.beta_1 = 8      # coefficient on log(X1) or X1
        self.beta_2 = 3      # coefficient on X2
        self.beta_U = 2      # coefficient on U
        self.staff_loading = 5  # U -> X2 coefficient

    def generate(self, n=500, seed=42):
        """Generate observed data: revenue, ad_spend, staff_count, satisfaction."""
        rng = np.random.default_rng(seed)

        # Step 1: Generate U, noise terms, epsilon base
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)

        # Step 2: X1, X2 from U
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01  # Ensure positive for log
        X2 = self.staff_loading * U + eta2

        # Step 3: Compute Y
        # Heteroscedastic errors: eps ~ N(0, h * X1)
        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)

        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps

        # Step 4: X3 from Y (post-treatment / bad control)
        X3 = self.bad_control_strength * Y + eta3

        return pd.DataFrame({
            'revenue': Y,
            'ad_spend': X1,
            'staff_count': X2,
            'satisfaction': X3,
        })

    def truth(self, n=500, seed=42):
        """Generate data including hidden demand_U + return true parameter dict."""
        rng = np.random.default_rng(seed)

        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)

        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2

        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)

        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps

        X3 = self.bad_control_strength * Y + eta3

        df = pd.DataFrame({
            'revenue': Y,
            'ad_spend': X1,
            'staff_count': X2,
            'satisfaction': X3,
            'demand_U': U,
        })

        params = {
            'beta_0': self.beta_0,
            'beta_1': self.beta_1,
            'beta_2': self.beta_2,
            'beta_U': self.beta_U,
            'sigma_epsilon': self.heteroscedasticity_strength,
        }

        return df, params

    def dgp_summary(self):
        """Return LaTeX specification string."""
        func = r"\log(X_1)" if self.nonlinearity else r"X_1"
        return (
            rf"$Y = {self.beta_0} + {self.beta_1} \cdot {func} "
            rf"+ {self.beta_2} \cdot X_2 + {self.beta_U} \cdot U + \varepsilon$"
            "\n\nWhere:\n"
            rf"- $U \sim N(0, 1)$ (unobserved market demand)"
            "\n"
            rf"- $X_1 = {self.endogeneity_strength} \cdot U + \eta_1$ (ad spend)"
            "\n"
            rf"- $X_2 = {self.staff_loading} \cdot U + \eta_2$ (staff count)"
            "\n"
            rf"- $X_3 = {self.bad_control_strength} \cdot Y + \eta_3$ (satisfaction, post-treatment)"
            "\n"
            rf"- $\varepsilon \sim N(0, {self.heteroscedasticity_strength} \cdot X_1)$"
            "\n"
            rf"- $\eta_i \sim N(0, {self.noise_sigma}^2)$"
        )

    def set_heteroscedasticity(self, strength):
        """Update heteroscedasticity strength."""
        self.heteroscedasticity_strength = strength

    def set_endogeneity(self, strength):
        """Update endogeneity strength (U -> X1 coefficient)."""
        self.endogeneity_strength = strength

    def set_nonlinearity(self, curvature):
        """Toggle or adjust nonlinearity. Pass bool or truthy value."""
        self.nonlinearity = bool(curvature)


# ---------------------------------------------------------------------------
# MonteCarloEngine — precompute simulation results for interactive sliders
# ---------------------------------------------------------------------------
class MonteCarloEngine:
    """Precomputes Monte Carlo draws across a parameter grid."""

    def run(self, estimator_fn, param_name, param_grid, simulator,
            n_reps=5000, n_obs=500):
        """
        For each value in param_grid, set param on simulator, run n_reps
        replications, collect estimator_fn output.

        Returns numpy array of shape (len(param_grid), n_reps).
        """
        results = np.empty((len(param_grid), n_reps))

        # Progress bar
        try:
            progress = widgets.IntProgress(
                value=0, min=0, max=len(param_grid),
                description='Simulating:', style={'description_width': 'initial'},
            )
            from IPython.display import display
            display(progress)
            use_widget = True
        except Exception:
            use_widget = False

        setter = getattr(simulator, f'set_{param_name}', None)

        for i, val in enumerate(param_grid):
            if setter is not None:
                setter(val)
            for r in range(n_reps):
                data = simulator.generate(n=n_obs, seed=r)
                results[i, r] = estimator_fn(data)
            if use_widget:
                progress.value = i + 1
            else:
                print(f"  [{i+1}/{len(param_grid)}] {param_name}={val:.3f}")

        if use_widget:
            progress.bar_style = 'success'

        return results

    def quick_run(self, estimator_fn, dgp_fn, n_reps=1000, n_obs=500):
        """
        Single-configuration simulation for sidebar mini-sims.
        dgp_fn(seed, n) -> DataFrame. No caching.
        """
        results = np.empty(n_reps)
        for r in range(n_reps):
            data = dgp_fn(seed=r, n=n_obs)
            results[r] = estimator_fn(data)
        return results


# ---------------------------------------------------------------------------
# DiagnosticSuite — standardized diagnostic visualizations
# ---------------------------------------------------------------------------
class DiagnosticSuite:
    """Produces four-panel residual diagnostics and summary test statistics."""

    @staticmethod
    def run_diagnostics(model_result):
        """
        2x2 diagnostic plot grid from a statsmodels OLS results object.
        Returns the matplotlib Figure.
        """
        fig, axes = plt.subplots(2, 2, figsize=(10, 8))

        influence = OLSInfluence(model_result)
        fitted = model_result.fittedvalues
        resid = model_result.resid
        std_resid = influence.resid_studentized_internal

        # Top-left: Residuals vs Fitted
        ax = axes[0, 0]
        ax.scatter(fitted, resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1.5)
        ax.set_xlabel('Fitted values')
        ax.set_ylabel('Residuals')
        ax.set_title('Residuals vs Fitted')

        # Top-right: QQ plot
        ax = axes[0, 1]
        stats.probplot(resid, dist="norm", plot=ax)
        ax.set_title('Normal Q-Q')
        ax.get_lines()[0].set_color(COLORS['ols'])
        ax.get_lines()[1].set_color(COLORS['bias'])

        # Bottom-left: Scale-Location
        ax = axes[1, 0]
        sqrt_abs_std = np.sqrt(np.abs(std_resid))
        ax.scatter(fitted, sqrt_abs_std, alpha=0.4, s=15, color=COLORS['ols'])
        ax.set_xlabel('Fitted values')
        ax.set_ylabel(r'$\sqrt{|Standardized\ Residuals|}$')
        ax.set_title('Scale-Location')

        # Bottom-right: Residuals vs Leverage
        ax = axes[1, 1]
        leverage = influence.hat_matrix_diag
        cooks_d = influence.cooks_distance[0]
        ax.scatter(leverage, std_resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1)
        ax.set_xlabel('Leverage')
        ax.set_ylabel('Standardized Residuals')
        ax.set_title("Residuals vs Leverage")

        # Cook's distance contours
        x_range = np.linspace(0.001, ax.get_xlim()[1], 100)
        for cook_val in [0.5, 1.0]:
            for sign in [1, -1]:
                p = model_result.df_model + 1
                y_val = sign * np.sqrt(cook_val * p * (1 - x_range) / x_range)
                ax.plot(x_range, y_val, '--', color=COLORS['bias'],
                        alpha=0.5, label=f"Cook's d={cook_val}" if sign == 1 else None)
        ax.legend(fontsize=8)

        fig.tight_layout()
        return fig

    @staticmethod
    def summary_tests(model_result):
        """
        Return dict of diagnostic test results:
        vif, breusch_pagan, durbin_watson, jarque_bera.
        """
        results = {}

        # VIF — requires exog with constant
        exog = model_result.model.exog
        exog_names = model_result.model.exog_names
        vif_dict = {}
        for i, name in enumerate(exog_names):
            if name == 'const':
                continue
            vif_dict[name] = variance_inflation_factor(exog, i)
        results['vif'] = vif_dict

        # Breusch-Pagan
        bp_stat, bp_pval, _, _ = het_breuschpagan(
            model_result.resid, model_result.model.exog
        )
        results['breusch_pagan'] = (bp_stat, bp_pval)

        # Durbin-Watson
        results['durbin_watson'] = durbin_watson(model_result.resid)

        # Jarque-Bera
        jb_stat, jb_pval, _, _ = jarque_bera(model_result.resid)
        results['jarque_bera'] = (jb_stat, jb_pval)

        return results


# ---------------------------------------------------------------------------
# AutopsyReport — widget factory for structured reflection
# ---------------------------------------------------------------------------
class AutopsyReport:
    """Creates reflection widgets for notebook wrap-up cells."""

    @staticmethod
    def lightweight(notebook_number):
        """Two-question mini autopsy for Notebooks 3-6."""
        threat = widgets.Text(
            description='Biggest threat:',
            placeholder='What is the biggest threat to this estimate?',
            layout=widgets.Layout(width='90%'),
            style={'description_width': '120px'},
        )
        check = widgets.Text(
            description='How to check:',
            placeholder='How would you check if that threat is real?',
            layout=widgets.Layout(width='90%'),
            style={'description_width': '120px'},
        )
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()

        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                record_trap_response(nb_id, "threat", threat.value)
                record_trap_response(nb_id, "check", check.value)
                print("Autopsy responses saved.")
                submit.disabled = True

        submit.on_click(on_submit)
        return widgets.VBox([
            widgets.HTML(f"<h3>Mini Autopsy — Notebook {notebook_number}</h3>"),
            threat, check, submit, output
        ])

    @staticmethod
    def full(notebook_number, available_sidebars=None):
        """Full sensitivity-analysis autopsy for Notebooks 7-8."""
        fields = {
            'point_estimate': widgets.Text(
                description='Point estimate:',
                placeholder='My point estimate is:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
            'robustness': widgets.Text(
                description='Robustness value:',
                placeholder='The robustness value is:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
            'partial_r2': widgets.Text(
                description='Strongest partial R²:',
                placeholder='The strongest observed covariate has partial R² of:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
            'plain_language': widgets.Text(
                description='Plain language:',
                placeholder='An omitted variable would need to be ___ times as important as ___ to explain away this result',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
        }

        children = [
            widgets.HTML(f"<h3>Full Autopsy Report — Notebook {notebook_number}</h3>"),
        ]
        children.extend(fields.values())

        if available_sidebars:
            sidebar_dropdown = widgets.Dropdown(
                options=['(select)'] + list(available_sidebars),
                description='Most analogous disaster:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '180px'},
            )
            children.append(sidebar_dropdown)
            fields['analogous_disaster'] = sidebar_dropdown

        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()

        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                for key, w in fields.items():
                    record_trap_response(nb_id, key, w.value)
                print("Full autopsy report saved.")
                submit.disabled = True

        submit.on_click(on_submit)
        children.extend([submit, output])
        return widgets.VBox(children)

# ── End embedded company_sim.py ──

from IPython.display import display, HTML, Markdown
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
%matplotlib inline

# ── NB1-specific DGP ──
NB1_BETA1_TRUE = 2.3
NB1_BETA2_TRUE = 2.0
NB1_DELTA = 1.2

class NB1Simulator:
    """Simplified linear DGP for clean OVB demonstration."""
    def __init__(self, rho=0.6):
        self.rho = rho

    def generate(self, n=500, seed=42, include_U=False):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        X1 = self.rho * U + rng.standard_normal(n) * 0.5
        X2 = U + rng.standard_normal(n) * 0.3
        eps = rng.standard_normal(n) * 1.0
        Y = 50 + NB1_BETA1_TRUE * X1 + NB1_BETA2_TRUE * X2 + eps
        X3 = 0.1 * Y + 0.5 * X1 + rng.standard_normal(n) * 0.5
        df = pd.DataFrame({
            'revenue': Y, 'ad_spend': X1,
            'staff_count': X2, 'satisfaction': X3,
        })
        if include_U:
            df['demand_U'] = U
        return df

    def set_rho(self, rho):
        self.rho = rho

nb1_sim = NB1Simulator(rho=0.6)

# ── Precompute Monte Carlo cache ──
print("Precomputing Monte Carlo simulations...")

def biased_estimator(data):
    X = sm.add_constant(data['ad_spend'])
    return sm.OLS(data['revenue'], X).fit().params['ad_spend']

rho_grid = np.round(np.arange(-0.95, 1.00, 0.1), 2)
N_REPS = 5000

mc_cache_rho = {}
progress = widgets.IntProgress(value=0, min=0, max=len(rho_grid),
    description='Precomputing:', style={'description_width': 'initial'})
display(progress)

for i, rho_val in enumerate(rho_grid):
    sim_temp = NB1Simulator(rho=rho_val)
    draws = np.empty(N_REPS)
    for r in range(N_REPS):
        data = sim_temp.generate(n=500, seed=r)
        draws[r] = biased_estimator(data)
    mc_cache_rho[round(rho_val, 2)] = draws
    progress.value = i + 1

progress.bar_style = 'success'

# Repair caches
def unbiased_estimator(data):
    X = sm.add_constant(data[['ad_spend', 'demand_U']])
    return sm.OLS(data['revenue'], X).fit().params['ad_spend']

def collider_estimator(data):
    X = sm.add_constant(data[['ad_spend', 'satisfaction']])
    return sm.OLS(data['revenue'], X).fit().params['ad_spend']

mc_cache_repair = {}
mc_cache_collider = {}
progress2 = widgets.IntProgress(value=0, min=0, max=len(rho_grid),
    description='Repair cache:', style={'description_width': 'initial'})
display(progress2)

for i, rho_val in enumerate(rho_grid):
    sim_temp = NB1Simulator(rho=rho_val)
    draws_u = np.empty(N_REPS)
    draws_c = np.empty(N_REPS)
    for r in range(N_REPS):
        data = sim_temp.generate(n=500, seed=r, include_U=True)
        draws_u[r] = unbiased_estimator(data)
        draws_c[r] = collider_estimator(data)
    mc_cache_repair[round(rho_val, 2)] = draws_u
    mc_cache_collider[round(rho_val, 2)] = draws_c
    progress2.value = i + 1

progress2.bar_style = 'success'

# Proxy quality cache for Limit cell
proxy_grid = np.arange(0, 105, 5)

def proxy_estimator(data, quality):
    rng = np.random.default_rng(99)
    noise_std = max(1e-6, (1.0 - quality/100.0) * 3.0)
    data = data.copy()
    data['proxy_U'] = data['demand_U'] + rng.standard_normal(len(data)) * noise_std
    X = sm.add_constant(data[['ad_spend', 'proxy_U']])
    return sm.OLS(data['revenue'], X).fit().params['ad_spend']

mc_cache_proxy = {}
progress3 = widgets.IntProgress(value=0, min=0, max=len(proxy_grid),
    description='Proxy cache:', style={'description_width': 'initial'})
display(progress3)

sim_base = NB1Simulator(rho=0.6)
for i, q in enumerate(proxy_grid):
    draws_p = np.empty(N_REPS)
    for r in range(N_REPS):
        data = sim_base.generate(n=500, seed=r, include_U=True)
        draws_p[r] = proxy_estimator(data, q)
    mc_cache_proxy[int(q)] = draws_p
    progress3.value = i + 1

progress3.bar_style = 'success'
print("\nSetup complete! All caches ready.")


# Notebook 1: Why Your Coefficient Is Wrong

> **Regression is the most dangerous tool in the data scientist's toolkit.** Not because it's complicated — because it *feels* simple. You run a regression, you get a coefficient, you make a decision. But the coefficient you see is almost never the coefficient you think it is. This notebook will show you why.

---

## The NovaMart Scenario

You are the newly hired data analyst at **NovaMart**, a mid-size retailer. The CEO asks a simple question: *"How much does advertising spend increase our revenue?"*

You have data on 500 stores: their **revenue**, **ad spend**, **staff count**, and **customer satisfaction** scores. You know how to run a regression. So you do.

In [ ]:
# ── The Trap: Simple regression of revenue on ad_spend ──
data = nb1_sim.generate(n=500, seed=42)

X = sm.add_constant(data['ad_spend'])
model = sm.OLS(data['revenue'], X).fit()
print(model.summary())

print("\n" + "="*60)
print(f"  Estimated coefficient on ad_spend: {model.params['ad_spend']:.2f}")
print("="*60)

# ── Trap Widget ──
trap = create_trap_widget(
    notebook_id="notebook_1",
    question_id="q1",
    question_text="Based on this regression, what would you tell the CEO?",
    options=[
        'A: "Ad spend strongly increases revenue. Recommend doubling budget."',
        'B: "Positive relationship but not confident it\'s causal."',
        'C: "Coefficient is biased upward — true effect is smaller."',
        'D: "Coefficient is biased but can\'t determine direction."',
    ]
)
display(trap)

In [ ]:
# ── The Reveal ──
if not check_gate("notebook_1", "q1"):
    display(HTML('<div style="background:#FFCCCC;padding:15px;border-radius:8px;">'
                 '<b>⚠️ Please answer the question in Cell 2 before continuing.</b></div>'))
else:
    response = get_trap_response("notebook_1", "q1")
    print(f"Your answer: {response}\n")

    if response.startswith("A"):
        display(HTML('<div style="background:#FFDDDD;padding:12px;border-radius:8px;">'
            '<b>Careful!</b> You fell into the classic trap. The coefficient is real, but it '
            "doesn't mean what you think. Keep reading.</div>"))
    elif response.startswith("B"):
        display(HTML('<div style="background:#FFF3CD;padding:12px;border-radius:8px;">'
            '<b>Cautious, but vague.</b> "Not causal" is hand-waving. Can you be precise '
            'about <i>what</i> is wrong and <i>which direction</i> the bias goes?</div>'))
    elif response.startswith("C"):
        display(HTML('<div style="background:#D4EDDA;padding:12px;border-radius:8px;">'
            "<b>Correct!</b> The coefficient is biased upward. Let's see exactly why.</div>"))
    elif response.startswith("D"):
        display(HTML('<div style="background:#FFF3CD;padding:12px;border-radius:8px;">'
            '<b>Close but wrong.</b> You <i>can</i> determine the direction. '
            'The OVB formula tells you exactly.</div>'))

    # ── Killer visualization: scatterplot colored by hidden U ──
    data_truth = nb1_sim.generate(n=500, seed=42, include_U=True)

    fig, ax = plt.subplots(figsize=(9, 6))
    sc = ax.scatter(data_truth['ad_spend'], data_truth['revenue'],
                    c=data_truth['demand_U'], cmap='RdYlBu_r',
                    alpha=0.7, s=30, edgecolors='gray', linewidth=0.3)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Hidden Demand (U)', fontsize=11)

    # Biased regression line
    x_line = np.linspace(data_truth['ad_spend'].min(), data_truth['ad_spend'].max(), 100)
    ax.plot(x_line, model.params['const'] + model.params['ad_spend'] * x_line,
            color=COLORS['bias'], linewidth=2.5, linestyle='--',
            label=f'Biased OLS: beta_hat = {model.params["ad_spend"]:.2f}')

    # True effect line
    mean_x = data_truth['ad_spend'].mean()
    mean_y = data_truth['revenue'].mean()
    ax.plot(x_line, mean_y + NB1_BETA1_TRUE * (x_line - mean_x),
            color=COLORS['truth'], linewidth=2.5, linestyle='-',
            marker='D', markevery=20, markersize=5,
            label=f'True effect: beta_1 = {NB1_BETA1_TRUE}')

    ax.set_xlabel('Ad Spend', fontsize=12)
    ax.set_ylabel('Revenue', fontsize=12)
    ax.set_title('The Hidden Variable: Stores with High Demand Spend More AND Earn More',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    fig.tight_layout()
    plt.show()

    # OVB formula
    display(HTML(
        '<div style="background:#F0F4FF;padding:20px;border-radius:10px;'
        'border-left:4px solid #2E5EA8;margin-top:15px;">'
        '<h3>Omitted Variable Bias Formula</h3>'
        '<p style="font-size:18px;">'
        r'$$E[\hat{\beta}_1] = \beta_1 + \beta_2 \cdot \delta_1$$'
        '</p><p>Where:</p><ul>'
        r'<li>$\beta_1 = 2.3$ (true effect of ad spend)</li>'
        r'<li>$\beta_2 = 2.0$ (effect of demand on revenue)</li>'
        r'<li>$\delta_1 = 1.2$ (coefficient from regressing demand on ad spend)</li>'
        '</ul>'
        '<p style="font-size:16px;">'
        r'$$E[\hat{\beta}_1] = 2.3 + 2.0 \times 1.2 = \boxed{4.7}$$'
        '</p>'
        '<p><b>The bias is $+2.4$. Your coefficient is more than double the true effect.</b></p>'
        '</div>'
    ))


In [ ]:
# ── Monte Carlo Confirmation ──
draws = mc_cache_rho[0.6]
true_val = NB1_BETA1_TRUE
predicted_bias = NB1_BETA1_TRUE + NB1_BETA2_TRUE * NB1_DELTA  # 4.7

fig_mc, ax_mc = plt.subplots(figsize=(9, 5))
bins = np.linspace(true_val - 1, predicted_bias + 2, 60)

skip_btn = widgets.Button(description='Skip Animation', button_style='warning')
skip_flag = [False]
def on_skip(btn):
    skip_flag[0] = True
    btn.disabled = True
skip_btn.on_click(on_skip)
display(skip_btn)

n_frames = 200
batch = 25

for frame in range(n_frames):
    if skip_flag[0]:
        break
    n_show = min((frame + 1) * batch, len(draws))
    ax_mc.clear()
    ax_mc.hist(draws[:n_show], bins=bins, density=True,
               color=COLORS['ols'], alpha=0.7, edgecolor='white')
    ax_mc.axvline(true_val, color=COLORS['truth'], linewidth=2.5,
                  label=f'True beta_1 = {true_val}')
    ax_mc.axvline(predicted_bias, color=COLORS['bias'], linewidth=2,
                  linestyle=':', label=f'E[beta_hat] = {predicted_bias:.1f}')
    ax_mc.set_xlabel('Estimated beta_hat_1', fontsize=11)
    ax_mc.set_ylabel('Density', fontsize=11)
    ax_mc.set_title(f'Monte Carlo ({n_show:,} / {len(draws):,} replications)', fontsize=12)
    ax_mc.legend(fontsize=9)
    fig_mc.canvas.draw()
    fig_mc.canvas.flush_events()

# Final full histogram
ax_mc.clear()
ax_mc.hist(draws, bins=bins, density=True,
           color=COLORS['ols'], alpha=0.7, edgecolor='white')
ax_mc.axvline(true_val, color=COLORS['truth'], linewidth=2.5,
              label=f'True beta_1 = {true_val}')
ax_mc.axvline(predicted_bias, color=COLORS['bias'], linewidth=2,
              linestyle=':', label=f'E[beta_hat] = {predicted_bias:.1f}')
sim_mean = draws.mean()
ax_mc.axvline(sim_mean, color=COLORS['ols'], linewidth=2,
              linestyle='--', label=f'Simulated mean = {sim_mean:.2f}')
ax_mc.set_xlabel('Estimated beta_hat_1', fontsize=11)
ax_mc.set_ylabel('Density', fontsize=11)
ax_mc.set_title(f'Monte Carlo Complete ({len(draws):,} replications)',
                fontsize=12, fontweight='bold')
ax_mc.legend(fontsize=9)
fig_mc.tight_layout()
plt.show()

print(f"\n  Theory predicts E[beta_hat_1] = {predicted_bias:.2f}")
print(f"  Simulation gives E[beta_hat_1] = {sim_mean:.2f}")
print(f"  Match: {'✓' if abs(predicted_bias - sim_mean) < 0.1 else '✗'} (difference = {abs(predicted_bias - sim_mean):.4f})")


---
## Discussion Prompt

> **Your colleague says: "I controlled for every variable in the dataset so there can't be OVB." What's wrong with this reasoning?**

Think about:
- Can you observe every relevant variable?
- What does "every variable in the dataset" actually guarantee?
- Could controlling for *more* variables actually make things *worse*?

In [ ]:
# ── The Destruction Playground: rho slider ──
fig_dest, (ax_hist, ax_summ) = plt.subplots(1, 2, figsize=(13, 5),
    gridspec_kw={'width_ratios': [2, 1]})
plt.subplots_adjust(wspace=0.35)

rho_slider = widgets.SelectionSlider(
    options=[round(v, 2) for v in np.arange(-0.95, 1.00, 0.1)],
    value=0.6,
    description='rho (U-X1):',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
)

bins_dest = np.linspace(-2, 10, 60)

def update_destruction(change):
    rho_val = change['new'] if isinstance(change, dict) else change
    draws = mc_cache_rho[round(rho_val, 2)]

    ax_hist.clear()
    ax_hist.hist(draws, bins=bins_dest, density=True,
                 color=COLORS['ols'], alpha=0.7, edgecolor='white')
    ax_hist.axvline(NB1_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5,
                    label=f'True beta_1 = {NB1_BETA1_TRUE}')
    ax_hist.axvline(draws.mean(), color=COLORS['bias'], linewidth=2, linestyle=':',
                    label=f'Mean beta_hat = {draws.mean():.2f}')
    ax_hist.set_xlabel('Estimated beta_hat_1', fontsize=11)
    ax_hist.set_ylabel('Density', fontsize=11)
    ax_hist.set_title(f'Sampling Distribution (rho = {rho_val:.2f})',
                      fontsize=12, fontweight='bold')
    ax_hist.legend(fontsize=9)

    ax_summ.clear()
    ax_summ.axis('off')
    bias = draws.mean() - NB1_BETA1_TRUE
    bias_color = COLORS['bias'] if abs(bias) > 0.2 else COLORS['repair']
    summary_text = (
        f"rho = {rho_val:.2f}\n\n"
        f"True beta_1 = {NB1_BETA1_TRUE}\n"
        f"Mean beta_hat = {draws.mean():.2f}\n"
        f"Bias = {bias:+.2f}\n"
        f"Std Dev = {draws.std():.2f}\n\n"
        f"{'WARNING: BIASED' if abs(bias) > 0.2 else 'Approximately unbiased'}"
    )
    ax_summ.text(0.1, 0.5, summary_text, fontsize=13,
                 verticalalignment='center', fontfamily='monospace',
                 bbox=dict(boxstyle='round,pad=0.8', facecolor='#F8F8F8',
                           edgecolor=bias_color, linewidth=2))
    ax_summ.set_title('Summary', fontsize=12, fontweight='bold')

    fig_dest.canvas.draw_idle()

rho_slider.observe(update_destruction, names='value')
update_destruction({'new': 0.6})
display(rho_slider)
plt.show()


In [ ]:
# ── The Repair: Toggle controls ──
fig_rep, axes_rep = plt.subplots(1, 3, figsize=(16, 5))
plt.subplots_adjust(wspace=0.35)

toggle_demand = widgets.ToggleButton(value=False,
    description='Include demand (U) as control',
    button_style='', layout=widgets.Layout(width='250px'))
toggle_collider = widgets.ToggleButton(value=False,
    description='Include satisfaction (X3) as control',
    button_style='', layout=widgets.Layout(width='280px'))

rho_repair = widgets.SelectionSlider(
    options=[round(v, 2) for v in np.arange(-0.95, 1.00, 0.1)],
    value=0.6, description='rho:', style={'description_width': '30px'},
    layout=widgets.Layout(width='400px'))

bins_rep = np.linspace(-2, 10, 60)

def update_repair(change=None):
    rho_val = round(rho_repair.value, 2)

    for ax in axes_rep:
        ax.clear()

    # Left: biased
    draws_biased = mc_cache_rho[rho_val]
    axes_rep[0].hist(draws_biased, bins=bins_rep, density=True,
                     color=COLORS['ols'], alpha=0.7, edgecolor='white')
    axes_rep[0].axvline(NB1_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5)
    axes_rep[0].set_title(f'Biased (omit U)\nMean={draws_biased.mean():.2f}', fontsize=11)
    axes_rep[0].set_xlabel('beta_hat_1')
    axes_rep[0].set_ylabel('Density')

    # Middle: with demand control
    if toggle_demand.value:
        draws_fixed = mc_cache_repair[rho_val]
        axes_rep[1].hist(draws_fixed, bins=bins_rep, density=True,
                         color=COLORS['repair'], alpha=0.7, edgecolor='white')
        axes_rep[1].axvline(NB1_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5)
        axes_rep[1].set_title(f'Repaired (include U)\nMean={draws_fixed.mean():.2f}', fontsize=11)
        toggle_demand.button_style = 'success'
    else:
        axes_rep[1].text(0.5, 0.5, 'Toggle "Include demand"\nto activate',
                         ha='center', va='center', fontsize=12, color='gray',
                         transform=axes_rep[1].transAxes)
        axes_rep[1].set_title('Repaired (include U)', fontsize=11, color='gray')
        toggle_demand.button_style = ''
    axes_rep[1].set_xlabel('beta_hat_1')
    axes_rep[1].set_ylabel('Density')

    # Right: with collider
    if toggle_collider.value:
        draws_col = mc_cache_collider[rho_val]
        axes_rep[2].hist(draws_col, bins=bins_rep, density=True,
                         color=COLORS['bias'], alpha=0.7, edgecolor='white')
        axes_rep[2].axvline(NB1_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5)
        axes_rep[2].set_title(f'Collider bias! (include X3)\nMean={draws_col.mean():.2f}',
                              fontsize=11, color=COLORS['bias'])
        toggle_collider.button_style = 'danger'
    else:
        axes_rep[2].text(0.5, 0.5, 'Toggle "Include satisfaction"\nto activate',
                         ha='center', va='center', fontsize=12, color='gray',
                         transform=axes_rep[2].transAxes)
        axes_rep[2].set_title('Collider (include X3)', fontsize=11, color='gray')
        toggle_collider.button_style = ''
    axes_rep[2].set_xlabel('beta_hat_1')
    axes_rep[2].set_ylabel('Density')

    fig_rep.canvas.draw_idle()

toggle_demand.observe(update_repair, names='value')
toggle_collider.observe(update_repair, names='value')
rho_repair.observe(update_repair, names='value')
update_repair()
display(widgets.HBox([toggle_demand, toggle_collider]))
display(rho_repair)
plt.show()

# ── DAG visualization ──
fig_dag, ax_dag = plt.subplots(figsize=(6, 4))
ax_dag.set_xlim(-0.5, 2.5)
ax_dag.set_ylim(-0.5, 1.5)
ax_dag.axis('off')

# Nodes
node_pos = {'U\n(Demand)': (1, 1.2), 'X1\n(Ad Spend)': (0, 0), 'Y\n(Revenue)': (2, 0)}
for label, (x, y) in node_pos.items():
    ax_dag.annotate(label, (x, y), fontsize=11, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8E8E8', edgecolor='black'))

# Arrows
arrows = [((1, 1.0), (0.2, 0.2)), ((1, 1.0), (1.8, 0.2)), ((0.3, 0.0), (1.7, 0.0))]
for start, end in arrows:
    ax_dag.annotate('', xy=end, xytext=start,
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax_dag.set_title('Causal DAG: Omitted Variable Bias', fontsize=12, fontweight='bold')
fig_dag.tight_layout()
plt.show()


In [ ]:
# ── The Limit: Proxy quality slider ──
fig_lim, (ax_lhist, ax_lcurve) = plt.subplots(1, 2, figsize=(13, 5),
    gridspec_kw={'width_ratios': [1.2, 1]})
plt.subplots_adjust(wspace=0.35)

proxy_slider = widgets.SelectionSlider(
    options=list(range(0, 105, 5)),
    value=50,
    description='Proxy quality %:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='500px'),
)

bins_lim = np.linspace(0, 6, 50)

# Precompute bias curve
proxy_means = {q: mc_cache_proxy[q].mean() for q in range(0, 105, 5)}
proxy_qs = sorted(proxy_means.keys())
proxy_biases = [proxy_means[q] - NB1_BETA1_TRUE for q in proxy_qs]

def update_limit(change=None):
    q = proxy_slider.value
    draws = mc_cache_proxy[int(q)]

    ax_lhist.clear()
    ax_lhist.hist(draws, bins=bins_lim, density=True,
                  color=COLORS['ols'], alpha=0.7, edgecolor='white')
    ax_lhist.axvline(NB1_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5,
                     label=f'True beta_1 = {NB1_BETA1_TRUE}')
    ax_lhist.axvline(draws.mean(), color=COLORS['ols'], linewidth=2, linestyle='--',
                     label=f'Mean = {draws.mean():.2f}')
    bias = draws.mean() - NB1_BETA1_TRUE
    status = "Repair effective" if abs(bias) < 0.3 else "Repair unreliable"
    marker = "checkmark" if abs(bias) < 0.3 else "warning"
    ax_lhist.set_title(f'{status} (proxy quality = {q}%)', fontsize=12, fontweight='bold',
                       color=COLORS['repair'] if abs(bias) < 0.3 else COLORS['bias'])
    ax_lhist.set_xlabel('beta_hat_1')
    ax_lhist.set_ylabel('Density')
    ax_lhist.legend(fontsize=9)

    ax_lcurve.clear()
    ax_lcurve.plot(proxy_qs, proxy_biases, color=COLORS['ols'], linewidth=2,
                   marker='o', markersize=4)
    ax_lcurve.axhline(0, color=COLORS['truth'], linewidth=1.5, linestyle='--')
    ax_lcurve.axvline(q, color=COLORS['bias'], linewidth=1.5, linestyle=':', alpha=0.7)
    ax_lcurve.set_xlabel('Proxy Quality (%)', fontsize=11)
    ax_lcurve.set_ylabel('Bias (beta_hat - beta_1)', fontsize=11)
    ax_lcurve.set_title('Partial Debiasing Curve', fontsize=12, fontweight='bold')

    fig_lim.canvas.draw_idle()

proxy_slider.observe(update_limit, names='value')
update_limit()
display(proxy_slider)
plt.show()


In [ ]:
#@title Real-World Disaster: The HRT Catastrophe

story_html = (
    '<div style="padding:15px;font-size:14px;line-height:1.6;">'
    '<h4>The Hormone Replacement Therapy Disaster</h4>'
    '<p>For decades, observational studies showed that women taking HRT had <b>lower</b> rates of '
    'heart disease. Doctors prescribed it to millions. Then randomized controlled trials (WHI, 2002) '
    'revealed HRT actually <b>increased</b> heart disease risk.</p>'
    '<p>What happened? The women who <i>chose</i> to take HRT were wealthier, more health-conscious, '
    'and had better diets — all of which independently reduced heart disease. The observational '
    'studies had massive <b>omitted variable bias</b> that flipped the sign of the coefficient.</p>'
    '</div>'
)

math_html = (
    '<div style="padding:15px;font-size:14px;line-height:1.8;">'
    '<h4>The OVB Math Behind the Sign Flip</h4>'
    r'<p>$$\beta_{\text{HRT}} = +0.5 \text{ (true harmful effect)}$$</p>'
    r'<p>$$\beta_U = -3.0 \text{ (healthy lifestyle reduces heart disease)}$$</p>'
    r'<p>$$\rho(\text{HRT}, U) = 0.6 \text{ (healthy women choose HRT)}$$</p>'
    r'<p>$$\delta \approx 0.6$$</p>'
    r'<p>$$E[\hat{\beta}] = 0.5 + (-3.0)(0.6) = 0.5 - 1.8 = \boxed{-1.3}$$</p>'
    '<p><b>The sign flips!</b> A harmful drug appears protective because of who takes it.</p>'
    '</div>'
)

sim_output = widgets.Output()
sim_btn = widgets.Button(description='Run HRT Simulation', button_style='primary')

def run_hrt_sim(btn):
    with sim_output:
        sim_output.clear_output(wait=True)
        engine = MonteCarloEngine()

        def hrt_dgp(seed, n):
            rng = np.random.default_rng(seed)
            U = rng.standard_normal(n)
            HRT = 0.6 * U + rng.standard_normal(n) * 0.5
            heart_disease = 0.5 * HRT - 3.0 * U + rng.standard_normal(n)
            return pd.DataFrame({'heart_disease': heart_disease, 'HRT': HRT, 'lifestyle_U': U})

        def hrt_biased(data):
            X = sm.add_constant(data['HRT'])
            return sm.OLS(data['heart_disease'], X).fit().params['HRT']

        results = engine.quick_run(hrt_biased, hrt_dgp, n_reps=1000, n_obs=500)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(results, bins=50, density=True, color=COLORS['ols'], alpha=0.7, edgecolor='white')
        ax.axvline(0.5, color=COLORS['truth'], linewidth=2.5, label='True beta_HRT = +0.5 (harmful)')
        ax.axvline(-1.3, color=COLORS['bias'], linewidth=2, linestyle=':',
                   label='E[beta_hat] = -1.3 (appears protective!)')
        ax.axvline(results.mean(), color=COLORS['ols'], linewidth=2, linestyle='--',
                   label=f'Sim mean = {results.mean():.2f}')
        ax.axvline(0, color='black', linewidth=0.5, linestyle='-')
        ax.set_xlabel('Estimated beta_hat_HRT')
        ax.set_ylabel('Density')
        ax.set_title('HRT Disaster: OVB Flips the Sign', fontweight='bold')
        ax.legend(fontsize=9)
        fig.tight_layout()
        plt.show()
        print(f"  True effect: +0.5 (harmful)")
        print(f"  OVB prediction: -1.3 (appears protective)")
        print(f"  Simulation mean: {results.mean():.2f}")

sim_btn.on_click(run_hrt_sim)

tab = widgets.Tab(children=[
    widgets.HTML(story_html),
    widgets.HTML(math_html),
    widgets.VBox([sim_btn, sim_output]),
])
tab.set_title(0, 'Story')
tab.set_title(1, 'Math')
tab.set_title(2, 'Simulation')
display(tab)


In [ ]:
# ── Mini Autopsy ──
autopsy = AutopsyReport.lightweight(1)
display(autopsy)

---
## What's Next?

Your coefficient is correct now — you've controlled for the right variables and avoided the collider trap. **But can you trust the standard errors?**

The confidence interval around your estimate assumes something about the *spread* of the errors. What if that assumption is wrong?

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 12px; color: white; margin-top: 20px;">
<h3 style="color: white; margin-top: 0;">Notebook 2: Why Your Uncertainty Is Wrong</h3>
<p>Your coefficient is fixed. Now let's break your confidence intervals.</p>
<p><a href="02_uncertainty.ipynb" style="color: #FFD700; font-weight: bold;">Open Notebook 2</a></p>
</div>